# MCP from scratch: hands-on (optional)
**LLMs, Animated · Lesson AG.3**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

You'll build, in plain Python with no network and no API key:
- the **JSON-RPC 2.0** messages MCP is made of,
- a tiny **schema validator** for tool arguments,
- an **MCP-style server** that answers `tools/list` and `tools/call` for two toy tools,
- a **client loop** with a fake, rule-based "model" that picks a tool from the listed schemas.

It follows the shape of the current MCP specification (revision `2026-07-28`, which is stateless: every request carries its protocol version in `_meta`, and there is no `initialize` handshake). It is a teaching toy, not a complete implementation. An optional last section runs the same tool with the official MCP Python SDK.

In [ ]:
import json, re

PROTOCOL_VERSION = "2026-07-28"
META = {
    "io.modelcontextprotocol/protocolVersion": PROTOCOL_VERSION,
    "io.modelcontextprotocol/clientCapabilities": {},
}

_next_id = [0]
def make_request(method, params=None):
    """A JSON-RPC 2.0 request string, with MCP's per-request _meta attached."""
    _next_id[0] += 1
    p = dict(params or {})
    p["_meta"] = META
    return json.dumps({"jsonrpc": "2.0", "id": _next_id[0], "method": method, "params": p})

# Toy data the tools will use
ORDERS = {
    "A17": {"status": "shipped Tuesday", "eta": "Friday"},
    "B42": {"status": "being packed", "eta": "next Monday"},
}
RATES_FROM_USD = {"EUR": 0.92, "GBP": 0.79, "INR": 83.0}   # made-up, fixed rates
print("ready:", make_request("tools/list"))

## 1 · JSON-RPC responses
Every MCP message is a JSON-RPC 2.0 object. A **response** repeats the request's `id` and has *either* a `result` *or* an `error`:

```json
{"jsonrpc": "2.0", "id": 7, "result": {...}}
{"jsonrpc": "2.0", "id": 7, "error": {"code": -32601, "message": "Method not found"}}
```
Since revision 2026-07-28, every MCP result also carries a `resultType` field: `"complete"` for a finished result (or `"input_required"` when the server needs more from the user). Our toy only ever returns `"complete"`.

Write `make_result(id, result)` and `make_error(id, code, message)`. Both return a JSON **string**. `make_result` must add `"resultType": "complete"` to the result dict (without changing the caller's dict).

In [ ]:
def make_result(id, result):
    # TODO
    raise NotImplementedError("Exercise 1")

def make_error(id, code, message):
    # TODO
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
r = json.loads(make_result(3, {"tools": []}))
assert r == {"jsonrpc": "2.0", "id": 3, "result": {"tools": [], "resultType": "complete"}}, r
orig = {"x": 1}; make_result(1, orig)
assert orig == {"x": 1}, "don't modify the caller's dict"
e = json.loads(make_error(4, -32601, "Method not found"))
assert e == {"jsonrpc": "2.0", "id": 4, "error": {"code": -32601, "message": "Method not found"}}, e
assert "result" not in e
print("✓ exercise 1")

## 2 · Validate tool arguments against a schema
Each MCP tool publishes an `inputSchema` in **JSON Schema**. A server should reject bad arguments before running anything. Implement a small subset in `validate(schema, args)`, returning a **list of error strings** (empty if valid):

- `args` must be a dict (`type: object`);
- every name in `required` must be present;
- if `additionalProperties` is `False`, no keys outside `properties` are allowed;
- for each present property, check `type`: `"string"`, `"number"` (int or float, but **not** bool), `"integer"` (int, not bool) or `"boolean"`;
- if the property has `enum`, the value must be one of them; if it has `pattern`, a string must match it fully (`re.fullmatch`); if it has `minimum`, a number must be ≥ it.

Any wording for the messages is fine; the checks only count them.

In [ ]:
def validate(schema, args):
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
S = {"type": "object",
     "properties": {"amount": {"type": "number", "minimum": 0},
                    "to": {"type": "string", "enum": ["EUR", "GBP", "INR"]},
                    "order_id": {"type": "string", "pattern": "[A-Z][0-9]+"}},
     "required": ["amount", "to"], "additionalProperties": False}
assert validate(S, {"amount": 10, "to": "EUR"}) == []
assert validate(S, {"amount": 2.5, "to": "INR", "order_id": "A17"}) == []
assert len(validate(S, {"to": "EUR"})) == 1                      # missing amount
assert len(validate(S, {"amount": "10", "to": "EUR"})) == 1      # wrong type
assert len(validate(S, {"amount": True, "to": "EUR"})) == 1      # bool is not a number
assert len(validate(S, {"amount": 10, "to": "JPY"})) == 1        # not in enum
assert len(validate(S, {"amount": -1, "to": "EUR"})) == 1        # below minimum
assert len(validate(S, {"amount": 1, "to": "EUR", "order_id": "a17"})) == 1   # pattern
assert len(validate(S, {"amount": 1, "to": "EUR", "x": 1})) == 1 # additional property
assert validate(S, "not a dict") != []
print("✓ exercise 2")

## 3 · An MCP-style server
The `MiniServer` class below already knows how to **register** tools (the `tool` decorator stores each function with its name, description and input schema, in registration order). Your job is `handle(message)`: take one JSON-RPC request **string** and return one response **string**.

| Situation | Response |
|---|---|
| not valid JSON | error `-32700` (parse error), `id` `None` |
| missing `"jsonrpc": "2.0"` or `method` | error `-32600` (invalid request) |
| `params._meta` lacks `io.modelcontextprotocol/protocolVersion` | error `-32602` (invalid params) |
| `tools/list` | result `{"tools": [{"name", "description", "inputSchema"}, ...]}` in registration order |
| `tools/call` with an unknown tool name | error `-32602` |
| `tools/call` with arguments that fail `validate` | error `-32602`, message listing the problems |
| `tools/call` where the function raises | **result** with `isError: True` and the exception text as content |
| `tools/call` that works | result `{"content": [{"type": "text", "text": ...}], "isError": False}`; if the function returns a non-string, `json.dumps` it |
| any other method | error `-32601` (method not found) |

Note the split the real protocol makes: problems with the *request* are JSON-RPC **errors**; a tool that runs but fails reports `isError: True` inside a normal **result**, so the model can read what went wrong and try again.

In [ ]:
class MiniServer:
    def __init__(self, name):
        self.name = name
        self.tools = {}          # name -> {"description", "inputSchema", "fn"}; dicts keep insertion order

    def tool(self, name, description, input_schema):
        def register(fn):
            self.tools[name] = {"description": description, "inputSchema": input_schema, "fn": fn}
            return fn
        return register

    def handle(self, message):
        return handle(self, message)   # implemented below as a plain function, so you can edit it on its own

In [ ]:
def handle(server, message):
    # TODO
    raise NotImplementedError("Exercise 3")

### The two toy tools
Run this cell to create a server with two tools. Look at how each one is described: the `description` and `inputSchema` are **all** the model will ever see of it.

In [ ]:
server = MiniServer("shop")

@server.tool("get_order",
             "Look up the shipping status and delivery date of a customer order by its order id.",
             {"type": "object",
              "properties": {"order_id": {"type": "string", "pattern": "[A-Z][0-9]+",
                                          "description": "Order id such as A17"}},
              "required": ["order_id"], "additionalProperties": False})
def get_order(order_id):
    if order_id not in ORDERS:
        raise KeyError(f"no order {order_id}")
    o = ORDERS[order_id]
    return f"Order {order_id}: {o['status']}, arriving {o['eta']}."

@server.tool("convert_currency",
             "Convert an amount of money in US dollars into another currency (EUR, GBP or INR).",
             {"type": "object",
              "properties": {"amount": {"type": "number", "minimum": 0},
                             "to": {"type": "string", "enum": ["EUR", "GBP", "INR"]}},
              "required": ["amount", "to"], "additionalProperties": False})
def convert_currency(amount, to):
    return {"amount": round(amount * RATES_FROM_USD[to], 2), "currency": to}

print(list(server.tools))

**Check 3**

In [ ]:
def ask(method, params=None):
    return json.loads(server.handle(make_request(method, params)))

lst = ask("tools/list")
assert lst["result"]["resultType"] == "complete"
assert [t["name"] for t in lst["result"]["tools"]] == ["get_order", "convert_currency"]
assert set(lst["result"]["tools"][0]) == {"name", "description", "inputSchema"}

ok = ask("tools/call", {"name": "get_order", "arguments": {"order_id": "A17"}})
assert ok["result"]["isError"] is False and "shipped Tuesday" in ok["result"]["content"][0]["text"], ok
conv = ask("tools/call", {"name": "convert_currency", "arguments": {"amount": 100, "to": "EUR"}})
assert json.loads(conv["result"]["content"][0]["text"]) == {"amount": 92.0, "currency": "EUR"}, conv

assert ask("tools/call", {"name": "nope", "arguments": {}})["error"]["code"] == -32602
assert ask("tools/call", {"name": "get_order", "arguments": {"order_id": 17}})["error"]["code"] == -32602
fail = ask("tools/call", {"name": "get_order", "arguments": {"order_id": "Z99"}})
assert fail["result"]["isError"] is True and "Z99" in fail["result"]["content"][0]["text"], fail
assert ask("resources/list")["error"]["code"] == -32601
assert json.loads(server.handle("{not json"))["error"]["code"] == -32700
no_meta = json.dumps({"jsonrpc": "2.0", "id": 99, "method": "tools/list", "params": {}})
assert json.loads(server.handle(no_meta))["error"]["code"] == -32602
assert json.loads(server.handle(json.dumps({"id": 5, "method": "tools/list"})))["error"]["code"] == -32600
print("✓ exercise 3")

## 4 · A fake "model" that picks a tool
A real model reads the tool list and decides. Ours is a transparent rule: **score each tool by word overlap** between the user's message and the tool's `name` + `description`, and pick the highest score.

Write `pick_tool(message, tools)`:
- turn text into a set of lowercase words with `words(text)` (provided: letters only, length ≥ 3, minus a few stop-words; underscores in names count as spaces);
- score = number of shared words between the message and the tool's name + description;
- return the best tool **dict**, or `None` if the best score is 0. On a tie, keep the earlier tool (list order).

`extract_args(message, schema)` is provided: it fills each property from the message using the schema (enum values, numbers, regex patterns). That is the schema doing real work: the client knows *what* to look for without knowing anything about the tool.

In [ ]:
STOP = {"the", "and", "for", "with", "into", "what", "how", "much", "can", "you", "please", "its", "has", "are", "our", "this", "that", "from", "use", "when", "not", "any", "such"}

def words(text):
    return {w for w in re.findall(r"[a-z]+", text.lower().replace("_", " ")) if len(w) >= 3 and w not in STOP}

def extract_args(message, schema):
    args = {}
    for name, spec in schema.get("properties", {}).items():
        if "enum" in spec:
            hit = [v for v in spec["enum"] if re.search(rf"\b{re.escape(v)}\b", message, re.I)]
            if hit: args[name] = hit[0]
        elif spec.get("type") in ("number", "integer"):
            m = re.search(r"-?\d+(?:\.\d+)?", message)
            if m: args[name] = float(m.group()) if spec["type"] == "number" else int(float(m.group()))
        elif "pattern" in spec:
            m = re.search(rf"\b(?:{spec['pattern']})\b", message)
            if m: args[name] = m.group()
    return args

print(words("Where is my order A17? It hasn't shipped_yet"))

In [ ]:
def pick_tool(message, tools):
    # TODO
    raise NotImplementedError("Exercise 4")

**Check 4**

In [ ]:
tools = ask("tools/list")["result"]["tools"]
assert pick_tool("Where is my order A17? Has it shipped?", tools)["name"] == "get_order"
assert pick_tool("Convert 250 dollars to GBP please", tools)["name"] == "convert_currency"
assert pick_tool("Tell me a joke", tools) is None
t = pick_tool("How much is 40 US dollars in INR?", tools)
assert t["name"] == "convert_currency" and extract_args("How much is 40 US dollars in INR?", t["inputSchema"]) == {"amount": 40.0, "to": "INR"}
print("✓ exercise 4")

## 5 · The client loop
Now wire it together, the way a host does on every turn. Write `run_turn(server, message)`:
1. send `tools/list` (use `make_request` and `server.handle`, parsing the JSON);
2. `pick_tool`; if it returns `None`, return `"Sorry, I have no tool for that."`;
3. `extract_args` from the message with the tool's `inputSchema`;
4. send `tools/call`;
5. if the response has an `error`, return `"Request error: " + message`; if `isError` is true, return `"Tool failed: " + text`; otherwise return `"[<tool name>] " + text`.

In a real host, step 2 is the LLM, the result text goes back into its context, and the model writes the final answer.

In [ ]:
def run_turn(server, message):
    # TODO
    raise NotImplementedError("Exercise 5")

**Check 5**

In [ ]:
out = run_turn(server, "Where is my order B42?")
print(out)
assert out == "[get_order] Order B42: being packed, arriving next Monday.", out
out = run_turn(server, "Please convert 10 dollars to EUR")
print(out)
assert out.startswith("[convert_currency]") and '"amount": 9.2' in out, out
assert run_turn(server, "Write me a poem about cats") == "Sorry, I have no tool for that."
assert run_turn(server, "Where is order Q1?").startswith("Tool failed:")
assert run_turn(server, "Convert some dollars to EUR").startswith("Request error:")   # no amount found
print("✓ exercise 5")

## Experiment
1. **Add a tool, change no client code.** Register a third tool on `server` (say `get_weather(city)` with an `enum` of three cities and a fixed answer). Run `run_turn(server, "What's the weather in Paris?")`. The client discovered it through `tools/list` alone: that is the whole point of a protocol.
2. **Descriptions are the interface.** Change `get_order`'s description to something vague like `"Order helper."` and retry the order question. What happens to the pick? Real models are smarter than word overlap, but they too only know a tool through its name, description and schema.
3. **Tool poisoning, in miniature.** Give a tool a description stuffed with popular words ("order shipping dollars currency weather convert status…"). Watch it hijack requests meant for other tools. Real attacks hide *instructions* in descriptions; the defence starts with only installing servers you trust.

In [ ]:
# your experiments here

## Optional · the same tool with the official MCP Python SDK
The official SDK (`pip install mcp`) does all of the above for you: it builds the input schema from Python type hints, speaks JSON-RPC, and validates arguments. The cell below tries version 2 of the SDK (2026, `MCPServer`) and falls back to version 1 (`FastMCP`). It connects a client to the server **in memory**, so no network is needed.

If the package is missing it prints a note and skips. In Colab, uncomment the first line to install it (about 20 seconds).

In [ ]:
# !pip install -q mcp
try:
    import mcp
    HAVE_MCP = True
except ImportError:
    HAVE_MCP = False
    print("The `mcp` package isn't installed, so this section is skipped. In Colab: !pip install -q mcp")

async def sdk_demo():
    if not HAVE_MCP:
        return None
    try:                                            # SDK v2 (2026 onwards)
        from mcp.server import MCPServer
        from mcp import Client
        srv, v2 = MCPServer("shop"), True
    except ImportError:                             # SDK v1.x
        from mcp.server.fastmcp import FastMCP
        from mcp.shared.memory import create_connected_server_and_client_session
        srv, v2 = FastMCP("shop"), False

    @srv.tool()
    def get_order(order_id: str) -> str:
        """Look up the shipping status and delivery date of a customer order by its order id."""
        o = ORDERS[order_id]
        return f"Order {order_id}: {o['status']}, arriving {o['eta']}."

    if v2:
        async with Client(srv) as client:
            listed = await client.list_tools()
            result = await client.call_tool("get_order", {"order_id": "A17"})
    else:
        async with create_connected_server_and_client_session(srv._mcp_server) as client:
            listed = await client.list_tools()
            result = await client.call_tool("get_order", {"order_id": "A17"})
    tool = listed.tools[0]
    schema = getattr(tool, "input_schema", None) or getattr(tool, "inputSchema", None)
    print("SDK version:", "v2" if v2 else "v1")
    print("tools/list ->", tool.name, json.dumps(schema))
    print("tools/call ->", result.content[0].text)
    return result

sdk_result = await sdk_demo()

## Recap
- MCP messages are **JSON-RPC 2.0**: requests with `method`, `params` and `id`; responses with the same `id` and a `result` or an `error`.
- `tools/list` returns each tool's **name, description and input schema**; that is all a model knows about a tool.
- `tools/call` runs one tool. Bad requests are **errors**; a tool that runs and fails returns a **result with `isError: true`**.
- The client discovers tools at run time, so adding a tool to a server needs no client changes: N + M instead of N × M.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def make_result(id, result):
    body = dict(result)
    body["resultType"] = "complete"
    return json.dumps({"jsonrpc": "2.0", "id": id, "result": body})

def make_error(id, code, message):
    return json.dumps({"jsonrpc": "2.0", "id": id, "error": {"code": code, "message": message}})

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def validate(schema, args):
    if not isinstance(args, dict):
        return ["arguments must be an object"]
    errors = []
    props = schema.get("properties", {})
    for name in schema.get("required", []):
        if name not in args:
            errors.append(f"missing required argument: {name}")
    if schema.get("additionalProperties") is False:
        for name in args:
            if name not in props:
                errors.append(f"unexpected argument: {name}")
    checks = {
        "string": lambda v: isinstance(v, str),
        "number": lambda v: isinstance(v, (int, float)) and not isinstance(v, bool),
        "integer": lambda v: isinstance(v, int) and not isinstance(v, bool),
        "boolean": lambda v: isinstance(v, bool),
    }
    for name, value in args.items():
        spec = props.get(name)
        if spec is None:
            continue
        t = spec.get("type")
        if t in checks and not checks[t](value):
            errors.append(f"{name} must be of type {t}")
            continue
        if "enum" in spec and value not in spec["enum"]:
            errors.append(f"{name} must be one of {spec['enum']}")
        if "pattern" in spec and isinstance(value, str) and not re.fullmatch(spec["pattern"], value):
            errors.append(f"{name} does not match {spec['pattern']}")
        if "minimum" in spec and isinstance(value, (int, float)) and value < spec["minimum"]:
            errors.append(f"{name} must be >= {spec['minimum']}")
    return errors

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def handle(server, message):
    try:
        req = json.loads(message)
    except json.JSONDecodeError:
        return make_error(None, -32700, "Parse error")
    if not isinstance(req, dict) or req.get("jsonrpc") != "2.0" or "method" not in req:
        return make_error(req.get("id") if isinstance(req, dict) else None, -32600, "Invalid request")
    rid, method, params = req.get("id"), req["method"], req.get("params") or {}
    if "io.modelcontextprotocol/protocolVersion" not in params.get("_meta", {}):
        return make_error(rid, -32602, "Missing protocol version in _meta")
    if method == "tools/list":
        tools = [{"name": n, "description": t["description"], "inputSchema": t["inputSchema"]}
                 for n, t in server.tools.items()]
        return make_result(rid, {"tools": tools})
    if method == "tools/call":
        name, args = params.get("name"), params.get("arguments", {})
        if name not in server.tools:
            return make_error(rid, -32602, f"Unknown tool: {name}")
        tool = server.tools[name]
        problems = validate(tool["inputSchema"], args)
        if problems:
            return make_error(rid, -32602, "Invalid arguments: " + "; ".join(problems))
        try:
            out = tool["fn"](**args)
        except Exception as exc:
            return make_result(rid, {"content": [{"type": "text", "text": f"Error: {exc}"}], "isError": True})
        text = out if isinstance(out, str) else json.dumps(out)
        return make_result(rid, {"content": [{"type": "text", "text": text}], "isError": False})
    return make_error(rid, -32601, f"Method not found: {method}")

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def pick_tool(message, tools):
    msg = words(message)
    best, best_score = None, 0
    for t in tools:
        score = len(msg & words(t["name"] + " " + t["description"]))
        if score > best_score:
            best, best_score = t, score
    return best

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def run_turn(server, message):
    listed = json.loads(server.handle(make_request("tools/list")))
    tool = pick_tool(message, listed["result"]["tools"])
    if tool is None:
        return "Sorry, I have no tool for that."
    args = extract_args(message, tool["inputSchema"])
    resp = json.loads(server.handle(make_request("tools/call", {"name": tool["name"], "arguments": args})))
    if "error" in resp:
        return "Request error: " + resp["error"]["message"]
    text = resp["result"]["content"][0]["text"]
    if resp["result"]["isError"]:
        return "Tool failed: " + text
    return f"[{tool['name']}] {text}"